# Experimentos híbrido Polar + AE

Notebook de ejecución secuencial completa. Cada bloque tiene un objetivo claro,
se explica qué se busca demostrar y qué significan los resultados.

**Orden:** ejecutar de arriba a abajo sin saltarse celdas.

In [ ]:
# ── Setup: montar Drive, actualizar repo, imports ──────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO = '/content/drive/MyDrive/tfg-datos'
os.makedirs(REPO, exist_ok=True)
!git -C {REPO} pull 2>/dev/null || git clone https://github.com/monica793/tfg-datos.git {REPO}
os.chdir(REPO)
sys.path.insert(0, REPO)
print('Repo listo')

In [ ]:
# ── Dependencias ───────────────────────────────────────────────────────────
try:
    import sionna.phy
except ImportError:
    !pip install -q sionna

try:
    import wandb
    wandb.login()
except ImportError:
    !pip install -q wandb
    import wandb
    wandb.login()

In [ ]:
# ── Config global ──────────────────────────────────────────────────────────
# Modifica aquí si quieres cambiar parámetros sin tocar el código.
import training.train_hybrid as th

N         = 100
RHO_DBS   = [0.0, 3.0]
AE_EPOCHS = 20        # sube a 40-60 si tienes tiempo
AE_STEPS  = 300
P_EMPTY   = 0.3
K_REPR    = 50        # k representativo para plots de tau y latente

th.AE_EPOCHS          = AE_EPOCHS
th.AE_STEPS_PER_EPOCH = AE_STEPS

from training.train_hybrid import K_CAND, k_is_valid_for_5g
from utils.signal import rho_db_to_ebno_db

VALID_KS = [k for k in K_CAND if k < N and k_is_valid_for_5g(k, N) and not (12 <= k <= 19)]
print(f'k válidos ({len(VALID_KS)}): {VALID_KS}')

---
## Bloque A — Baseline

**Qué se hace:** entrena el modelo con la configuración actual y evalúa curvas P_FA, P_MD, P_IE vs tasa R=k/n.

**Qué se busca demostrar:** establecer el punto de partida del sistema, sin optimización de ningún tipo.

**Qué significan los resultados:**
- P_FA/P_MD planas y bajas → detección robusta e independiente de la tasa.
- P_IE sube con R → la decodificación empeora al reducir redundancia Polar (esperado).
- P_IE ≈ 0.70 para R alto → el sistema falla en casi todos los bloques activos (0.70 = P(A=1)).

In [ ]:
from training.train_hybrid import train_ae_for_n
from systems.hybrid_polar import ActivityAwarePolarSystem
from evaluation.plot_pfa_pmd_pie import run_curves_for_n

baseline_models = {}
for rho_db in RHO_DBS:
    baseline_models[rho_db] = train_ae_for_n(
        n=N, rho_db=rho_db, valid_ks=VALID_KS,
        checkpoint_tag='base', use_wandb=True
    )

for rho_db, ae in baseline_models.items():
    run_curves_for_n(
        n=N, rho_db=rho_db, k_cand=VALID_KS, label='Baseline',
        make_system=lambda k, n_: ActivityAwarePolarSystem(k=k, n=n_, ae_model=ae, p_empty=P_EMPTY)
    )

---
## Bloque B — Mezcla AE con señal original (alpha_mix)

**Qué se hace:** evalúa el mismo modelo entrenado, pero variando cuánto se usa la salida del AE
antes del demapper Polar. alpha=1.0 es el AE puro; alpha=0.0 es la señal recibida sin denoising.

**Qué se busca demostrar:** si el AE está perjudicando la decodificación Polar al reconstruir
símbolos con demasiada confianza (LLRs artificialmente grandes). Esto es una limitación
conocida de meter un denoiser 'duro' antes de un decoder blando.

**Qué significan los resultados:**
- Si el mínimo P_IE aparece en alpha < 1.0 → el AE puro está sobre-limpiando y perjudica.
- Si el mínimo está en alpha = 1.0 → el denoising ayuda o no perjudica.
- Si alpha=0.0 (sin AE) da P_IE similar → el AE no aporta nada a decodificación.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from evaluation.plot_pfa_pmd_pie import eval_pfa_pmd_pie

ALPHAS  = [0.0, 0.3, 0.5, 0.7, 1.0]
rho_db  = RHO_DBS[0]   # primero con el SNR más bajo (peor caso)
ae      = baseline_models[rho_db]
ebno_db = rho_db_to_ebno_db(rho_db, K_REPR / N)

pies = []
for alpha in ALPHAS:
    sys_ = ActivityAwarePolarSystem(k=K_REPR, n=N, ae_model=ae, p_empty=P_EMPTY, alpha_mix=alpha)
    _, pfa, pmd, pie = eval_pfa_pmd_pie(sys_, N, K_REPR, ebno_db)
    pies.append(pie)
    print(f'alpha={alpha:.1f} | P_FA={pfa:.2e} | P_MD={pmd:.2e} | P_IE={pie:.2e}')

plt.figure(figsize=(6, 4))
plt.semilogy(ALPHAS, pies, marker='o')
plt.xlabel('alpha_mix  (0=sin AE, 1=AE puro)'); plt.ylabel('P_IE')
plt.title(f'P_IE vs mezcla AE | rho={rho_db} dB | k={K_REPR}')
plt.grid(True, which='both', alpha=0.35)
os.makedirs('results/figures', exist_ok=True)
plt.savefig(f'results/figures/alpha_sweep_rho{rho_db}_k{K_REPR}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Mejor alpha: {ALPHAS[int(np.argmin(pies))]} -> P_IE={min(pies):.2e}')

---
## Bloque C — Sensibilidad a pesos de la función de pérdida

**Qué se hace:** entrena tres variantes del modelo cambiando solo el peso de la pérdida
de reconstrucción (w_recon) y de clasificación (w_class).

**Qué se busca demostrar:** que la elección w_class=10, w_recon=1 no es arbitraria,
y entender si equilibrar más la pérdida mejora la decodificación final.

**Qué significan los resultados:**
- Si 'recon_fuerte' baja P_IE → el AE necesitaba aprender mejor denoising, no solo detección.
- Si 'class_fuerte' baja P_FA/P_MD sin perjudicar P_IE → puede justificarse el peso alto.
- Si 'base' gana en todo → la configuración original es la mejor.

In [ ]:
from training.run_hybrid_experiments import run_all

WEIGHT_EXPS = [
    {'name': 'base',         'w_recon': 1.0, 'w_class': 10.0, 'latent_dim': 64, 'hidden_dim': 256},
    {'name': 'recon_fuerte', 'w_recon': 3.0, 'w_class': 10.0, 'latent_dim': 64, 'hidden_dim': 256},
    {'name': 'class_fuerte', 'w_recon': 1.0, 'w_class': 20.0, 'latent_dim': 64, 'hidden_dim': 256},
]

summary_C = run_all(experiments=WEIGHT_EXPS, n=N, rho_dbs=RHO_DBS,
                    ae_epochs=AE_EPOCHS, ae_steps=AE_STEPS, use_wandb=True)

---
## Bloque D — Arquitectura de red

**Qué se hace:** compara el modelo base con uno más grande (más neuronas en encoder/decoder).

**Qué se busca demostrar:** si la capacidad de la red es el cuello de botella,
o si el problema es estructural (interfaz AE-Polar, modelo de canal, etc.).

**Qué significan los resultados:**
- Si 'arq_grande' mejora P_IE → el modelo base es insuficiente.
- Si no mejora → la limitación no es capacidad de la red sino la arquitectura/interfaz.

In [ ]:
ARQ_EXPS = [
    {'name': 'base',      'w_recon': 1.0, 'w_class': 10.0, 'latent_dim': 64, 'hidden_dim': 256},
    {'name': 'arq_grande','w_recon': 1.0, 'w_class': 10.0, 'latent_dim': 96, 'hidden_dim': 384},
]

summary_D = run_all(experiments=ARQ_EXPS, n=N, rho_dbs=RHO_DBS,
                    ae_epochs=AE_EPOCHS, ae_steps=AE_STEPS, use_wandb=True)

---
## Bloque E — Calibración del umbral de actividad (tau)

**Qué se hace:** barre el umbral de decisión tau de 0.05 a 0.95 y genera
la curva ROC (P_MD vs P_FA) y P_IE vs tau.

**Qué se busca demostrar:** que tau=0.5 no es necesariamente óptimo,
y que existe un tau* que minimiza P_IE (o que cumple un objetivo de P_FA).

**Qué significan los resultados:**
- tau* < 0.5 → el modelo tiende a dar p_active baja incluso en activos (sube tau para ser más agresivo).
- tau* > 0.5 → el modelo es conservador, anuncia activo fácilmente (baja tau para ser más selectivo).
- Curva ROC amplia → margen de ajuste; curva estrecha → poco margen.

In [ ]:
from evaluation.plot_pfa_pmd_pie import run_threshold_sweep

# Usar el mejor modelo de bloque C (si se sabe) o el base
ae_for_tau = baseline_models[RHO_DBS[0]]

for rho_db in RHO_DBS:
    ae_for_tau = baseline_models[rho_db]
    ebno_db = rho_db_to_ebno_db(rho_db, K_REPR / N)
    run_threshold_sweep(
        n=N, k=K_REPR, ebno_db=ebno_db, label=f'Baseline_rho{rho_db}',
        make_system=lambda k, n_: ActivityAwarePolarSystem(
            k=k, n=n_, ae_model=ae_for_tau, p_empty=P_EMPTY
        )
    )

---
## Bloque F — Visualización del espacio latente

**Qué se hace:** proyecta las representaciones internas del AE a 2D (PCA)
y pinta cada bloque como un punto. El color indica si era activo/vacío;
el borde indica si el bloque falló globalmente (error P_IE: MD, FA o bits).

**Qué se busca demostrar:** que el AE aprendió una representación latente
que separa bien activos de vacíos, y que los errores globales (borde rojo)
no son aleatorios sino que se concentran en la frontera de decisión.

**Qué significan los resultados:**
- Dos nubes bien separadas → representación latente discriminativa (buena detección).
- Errores rojos en la frontera → el umbral tau separa bien; el fallo es decodificación, no detección.
- Errores rojos dispersos en el interior de nubes → fallo de decodificación, no de clasificación.

In [ ]:
from evaluation.latent_visualization import plot_latent_with_global_error

for rho_db in RHO_DBS:
    ae = baseline_models[rho_db]
    sys_ = ActivityAwarePolarSystem(k=K_REPR, n=N, ae_model=ae, p_empty=P_EMPTY)
    ebno_db = rho_db_to_ebno_db(rho_db, K_REPR / N)
    stats = plot_latent_with_global_error(
        system=sys_, ebno_db=ebno_db, batch_size=3000, n_batches=2,
        title=f'Espacio latente | rho={rho_db} dB | k={K_REPR}',
        save_path=f'results/figures/latent_rho{rho_db}_k{K_REPR}.png'
    )
    print(f'rho={rho_db} | error global = {stats["p_error_global"]:.3f}')

---
## Descarga de resultados

In [ ]:
import zipfile
from google.colab import files

ZIP = os.path.join(REPO, 'results_export.zip')
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['results/checkpoints', 'results/figures', 'results/experiments']:
        if os.path.isdir(folder):
            for fname in os.listdir(folder):
                zf.write(os.path.join(folder, fname), arcname=f'{folder}/{fname}')

print('ZIP creado:', ZIP)
files.download(ZIP)